# Calling qodec instructions from Q#

With `run_qir(..., qodec=...)`, each quantum call in the compiled QIR runs the instruction of the qodec's top instruction set whose mnemonic is **exactly** the callee's name. This notebook shows, one minimal example at a time, how to name and declare an instruction for each kind of Q# call.

The examples use one-layer qodecs whose instructions act directly on qubits, so each instruction needs only a declaration. The same naming rules apply to the top layer of an encoded qodec, where gadgets implement the instructions.

In [ ]:
from textwrap import indent

import qodec as qc
from qdk import qsharp, TargetProfile
from qdk.simulation import run_qir

# Q# compiles calls to custom measurements (section 5) only for an Adaptive target.
qsharp.init(target_profile=TargetProfile.Adaptive_RI)

PREPARE = """
- mnemonic: prepare
  description: Prepare |0>.
  out: [qubit]
  action: [stabilize: Z_0]
"""


def qodec_with(instructions: str) -> qc.Qodec:
    """A one-layer qodec with ``prepare`` plus the given instruction declarations."""
    return qc.Qodec.loads(f"""
---
qodec.yaml:
  name: tutorial
  layers:
  - instruction_set: tutorial.isa.yaml
---
tutorial.isa.yaml:
  name: tutorial
  blocks: {{qubit: 1}}
  instructions:
{indent(PREPARE + instructions, "  ")}
""")

Every instruction set below declares `prepare`. The runtime uses it to initialize each qubit before the qubit's first use. It finds `prepare` by its declared action, a positive Z stabilizer, not by name, so no Q# call names it.

## 1. Standard gates

A Q# standard gate compiles to a QIR call such as `__quantum__qis__x__body`, and that callee name is the mnemonic to declare:

| Q# | Mnemonic |
| :--- | :--- |
| `X(q)`, `H(q)`, `Rz(θ, q)` | `__quantum__qis__x__body`, `__quantum__qis__h__body`, `__quantum__qis__rz__body` |
| `CNOT(a, b)` | `__quantum__qis__cx__body` |
| `Adjoint S(q)` | `__quantum__qis__s__adj` |
| `M(q)`, `MResetZ(q)`, `Reset(q)` | `__quantum__qis__m__body`, `__quantum__qis__mresetz__body`, `__quantum__qis__reset__body` |

`print(qsharp.compile(...))` shows the callee names a program uses.

In [ ]:
X = """
- mnemonic: __quantum__qis__x__body
  description: Pauli X.
  in: [qubit]
  out: [qubit]
  action: [pauli: X_0]
"""
M = """
- mnemonic: __quantum__qis__m__body
  description: Measure Z.
  in: [qubit]
  action: [observe: Z_0]
"""

run_qir(qsharp.compile("{ use q = Qubit(); X(q); M(q) }"), qodec=qodec_with(X + M), shots=3)

A quantum call without an instruction of the same name fails before any shot runs:

In [ ]:
try:
    run_qir(qsharp.compile("{ use q = Qubit(); H(q); M(q) }"), qodec=qodec_with(X + M))
except ValueError as error:
    print(error)

## 2. Multi-qubit gates and adjoints

Qubit arguments bind the instruction's block operands in order, so `CNOT(a, b)` passes the control first. An adjoint compiles to an `__adj` callee instead of `__body`.

In [ ]:
CX = """
- mnemonic: __quantum__qis__cx__body
  description: CNOT.
  in: [qubit, qubit]
  out: [qubit, qubit]
  action: [clifford: {X_0: X_0 X_1, Z_1: Z_0 Z_1}]
"""
S_ADJ = """
- mnemonic: __quantum__qis__s__adj
  description: Adjoint S.
  in: [qubit]
  out: [qubit]
  action: [clifford: {X_0: -Y_0}]
"""

program = qsharp.compile("""{
    use (a, b) = (Qubit(), Qubit());
    X(a);
    CNOT(a, b);
    Adjoint S(b);
    [M(a), M(b)]
}""")
run_qir(program, qodec=qodec_with(X + M + CX + S_ADJ), shots=3)

## 3. Rotation angles and other classical arguments

Classical arguments bind the instruction's `parameters` in declaration order. A `Double` binds a `number`, an `Int` binds an `integer`, `number`, or `bit`, and a `Bool` binds a `boolean` or `bit`. `Rx(θ, q)` passes one `Double`:

In [ ]:
RX = """
- mnemonic: __quantum__qis__rx__body
  description: Rotation about X.
  in: [qubit]
  out: [qubit]
  parameters: {theta: number}
  action: [rotate: {pauli: X_0, angle: theta}]
"""

program = qsharp.compile("{ use q = Qubit(); Rx(Std.Math.PI() / 2.0, q); M(q) }")
results = run_qir(program, qodec=qodec_with(RX + M), shots=200, type="cpu")
{str(value): results.count(value) for value in set(results)}

## 4. Custom operations

A Q# operation with `body intrinsic` compiles to a call with the operation's own name, so the mnemonic is that name. Its classical arguments bind parameters just as in section 3.

In [ ]:
qsharp.eval("""
operation Flip(q : Qubit) : Unit { body intrinsic; }
operation Turn(angle : Double, q : Qubit) : Unit { body intrinsic; }
""")

CUSTOM = """
- mnemonic: Flip
  description: Pauli X under a custom name.
  in: [qubit]
  out: [qubit]
  action: [pauli: X_0]
- mnemonic: Turn
  description: Rotation about X by angle.
  in: [qubit]
  out: [qubit]
  parameters: {angle: number}
  action: [rotate: {pauli: X_0, angle: angle}]
"""

program = qsharp.compile("""{
    use (a, b) = (Qubit(), Qubit());
    Flip(a);
    Turn(Std.Math.PI(), b);
    [M(a), M(b)]
}""")
run_qir(program, qodec=qodec_with(CUSTOM + M), shots=3, type="cpu")

## 5. Custom measurements

A `@Measurement()` operation returns its `Result` through a result argument that follows its qubits. The instruction's `observe` action must declare one observable per returned `Result`.

In [ ]:
qsharp.eval("""
@Measurement()
operation MeasureX(q : Qubit) : Result { body intrinsic; }
""")

MEASURE_X = """
- mnemonic: MeasureX
  description: Measure X.
  in: [qubit]
  action: [observe: X_0]
"""

results = run_qir(
    qsharp.compile("{ use q = Qubit(); MeasureX(q) }"), qodec=qodec_with(MEASURE_X), shots=200
)
{str(value): results.count(value) for value in set(results)}

## 6. Several outcomes

A measurement that returns a tuple passes one result argument per element, in order. List one observable for each: `observe: [Z_0, Z_1]` reports two outcomes, while `observe: Z_0 Z_1` would report a single joint parity.

In [ ]:
qsharp.eval("""
@Measurement()
operation MeasureBoth(a : Qubit, b : Qubit) : (Result, Result) { body intrinsic; }
""")

MEASURE_BOTH = """
- mnemonic: MeasureBoth
  description: Measure Z on two qubits.
  in: [qubit, qubit]
  action: [observe: [Z_0, Z_1]]
"""

program = qsharp.compile("{ use (a, b) = (Qubit(), Qubit()); X(b); MeasureBoth(a, b) }")
run_qir(program, qodec=qodec_with(X + MEASURE_BOTH), shots=3)

## 7. Preparations

In [ ]:
qsharp.eval("operation PrepareOne(q : Qubit) : Unit { body intrinsic; }")

PREPARE_ONE = """
- mnemonic: PrepareOne
  description: Prepare |1>.
  out: [qubit]
  action: [stabilize: -Z_0]
"""

run_qir(
    qsharp.compile("{ use q = Qubit(); PrepareOne(q); M(q) }"),
    qodec=qodec_with(PREPARE_ONE + M),
    shots=3,
)

## 8. Flags

An instruction's `flags` report whether it succeeded, such as a verified preparation that raises `reject`. A call can treat them in either of two ways:

- **Not returned:** a call with no result argument for the flags leaves them to the runtime, which rejects any shot where a flag is raised. `on_shot_failure` decides what happens to that shot.
- **Returned:** a `@Measurement()` call with one extra `Result` per flag, after its outcomes, receives the flags. The program then decides what to do.

The instructions here act on qubits directly, so their flags always read `Zero`; in an encoded qodec, each gadget reports its own.

In [ ]:
qsharp.eval("""
operation PrepareChecked(q : Qubit) : Unit { body intrinsic; }

@Measurement()
operation PrepareCheckedWithFlag(q : Qubit) : Result { body intrinsic; }
""")

CHECKED = """
- mnemonic: PrepareChecked
  description: Prepare |0> and check it; raised flags reject the shot.
  out: [qubit]
  action: [stabilize: Z_0]
  flags: [reject]
- mnemonic: PrepareCheckedWithFlag
  description: Prepare |0> and check it; returns the flag.
  out: [qubit]
  action: [stabilize: Z_0]
  flags: [reject]
"""

program = qsharp.compile("""{
    use (a, b) = (Qubit(), Qubit());
    PrepareChecked(a);
    let flag = PrepareCheckedWithFlag(b);
    (flag, M(a), M(b))
}""")
run_qir(program, qodec=qodec_with(CHECKED + M), shots=3)

## Summary

| Q# call | Instruction to declare |
| :--- | :--- |
| Standard gate, such as `X(q)` or `Adjoint S(q)` | Mnemonic equal to its QIR callee, such as `__quantum__qis__x__body` or `__quantum__qis__s__adj` |
| `body intrinsic` operation `Foo` | Mnemonic `Foo` |
| Qubit arguments | Block operands, in order |
| `Double`, `Int`, and `Bool` arguments | `parameters`, in declaration order |
| `@Measurement()` returning *n* `Result` values | `observe` with *n* observables |
| … with one extra `Result` per flag | Flags returned to the program instead of rejecting the shot |
| First use that creates the qubit's block | Instruction with only `out` operands, replacing `prepare` |